<div align="center">
    <span style="font-size:30px">
        <strong>
            <!-- Símbolo de Python -->
            <img
                src="https://cdn3.emoji.gg/emojis/1887_python.png"
                style="margin-bottom:-5px"
                width="30px" 
                height="30px"
            >
            <!-- Título -->
            Python para Geólogos
            <!-- Versión -->
            <img 
                src="https://img.shields.io/github/release/kevinalexandr19/manual-python-geologia.svg?style=flat&label=&color=blue"
                style="margin-bottom:-2px" 
                width="40px"
            >
        </strong>
    </span>
    <br>
    <span>
        <!-- Github del proyecto -->
        <a href="https://github.com/kevinalexandr19/manual-python-geologia" target="_blank">
            <img src="https://img.shields.io/github/stars/kevinalexandr19/manual-python-geologia.svg?style=social&label=Github Repo">
        </a>
        &nbsp;&nbsp;
        <!-- Licencia -->
        <img src="https://img.shields.io/github/license/kevinalexandr19/manual-python-geologia.svg?color=forestgreen">
        &nbsp;&nbsp;
        <!-- Release date -->
        <img src="https://img.shields.io/github/release-date/kevinalexandr19/manual-python-geologia?color=gold">
    </span>
    <br>
    <span>
        <!-- Perfil de LinkedIn -->
        <a target="_blank" href="https://www.linkedin.com/in/kevin-alexander-gomez/">
            <img src="https://img.shields.io/badge/-Kevin Alexander Gomez-5eba00?style=social&logo=linkedin">
        </a>
        &nbsp;&nbsp;
        <!-- Perfil de Github -->
        <a target="_blank" href="https://github.com/kevinalexandr19">
            <img src="https://img.shields.io/github/followers/kevinalexandr19.svg?style=social&label=kevinalexandr19&maxAge=2592000">
        </a>
    </span>
    <br>
</div>

***

<span style="color:lightgreen; font-size:25px">**PG202 - Aprendizaje no supervisado**</span>

Bienvenido al curso!!!

Vamos a explorar qué significa encontrar grupos cuando tenemos una tabla sin etiquetas. Usaremos coordenadas en metros de una campaña de muestreo sintética.

<span style="color:gold; font-size:20px">**Clustering con K-means y DBSCAN**</span>
***

- [¿Qué aprende un modelo de clustering?](#parte-1)
- [K-means: distancia a los centros](#parte-2)
- [DBSCAN: vecindarios densos](#parte-3)
- [Comparación y revisión](#parte-4)

***

In [ ]:
import sys

if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "scikit-learn"])

<a id="parte-1"></a>

### <span style="color:lightgreen">**¿Qué aprende un modelo de clustering?**</span>
***

En [PCA](pg202a_pca.ipynb) cambiamos los ejes para resumir las variables. En **clustering** usamos las características de las filas para agrupar observaciones. Cada número de grupo identifica una agrupación del algoritmo. Para relacionarla con litologías o procesos hacen falta evidencias como mapas, ensayos o descripciones de campo.

Cada fila de `X` es una ubicación de muestreo, con este y norte en metros. Usaremos distancia euclidiana en el plano con igual escala en ambos ejes. Con variables de unidades distintas, la elección de la escala forma parte del modelo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score

# Dos trazas curvas y algunos puntos aislados, solo para la práctica.
trazas, _ = make_moons(n_samples=300, noise=0.055, random_state=42)
X = np.vstack([100 * trazas, [[-125, -62], [225, 115], [66, 146], [40, -104]]])
X_ajuste, X_reserva = train_test_split(X, test_size=0.20, random_state=42)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(*X_ajuste.T, s=21, color="#0072B2", label="Filas para ajustar")
ax.set(xlabel="Este (m)", ylabel="Norte (m)", title="Ubicaciones de muestreo sintéticas", aspect="equal")
ax.legend()
fig.tight_layout()
plt.show()

<a id="parte-2"></a>

### <span style="color:lightgreen">**K-means: distancia a los centros**</span>
***

K-means alterna dos pasos. Asigna cada fila al centro más cercano y recalcula cada centro con la media de las filas de su grupo. Esto reduce la suma de distancias cuadráticas dentro de los grupos, llamada **inercia**. El algoritmo parte de un número de grupos `k` elegido por nosotros y puede llegar a soluciones distintas según la inicialización.

En el ejemplo la hipótesis inicial es que hay dos grupos. Usaremos varias inicializaciones. Vamos a comprobar manualmente la decisión del modelo para una fila reservada.

In [ ]:
km = KMeans(n_clusters=2, n_init=10, random_state=42)
km.fit(X_ajuste)
centros = km.cluster_centers_
muestra = X_reserva[:1]
distancias = np.linalg.norm(centros - muestra, axis=1)
grupo_manual = distancias.argmin()
grupo_modelo = km.predict(muestra)[0]
print(f"Muestra en metros: {muestra.round(1)}")
print(f"Distancias a los centros: {distancias.round(1)} m")
print(f"Centro más cercano: {grupo_manual} | predict: {grupo_modelo}")

`fit` mueve los centros usando solo `X_ajuste`. `predict` usa esos centros fijos. Una fila muy lejana también recibe un grupo. Comparemos qué ocurre cuando la forma de los grupos importa.

<a id="parte-3"></a>

### <span style="color:lightgreen">**DBSCAN: vecindarios densos**</span>
***

DBSCAN busca puntos que tienen al menos `min_samples` observaciones dentro de una distancia `eps`, contando el propio punto. Esos **puntos núcleo** conectan regiones densas. Un punto de borde puede quedar en un grupo por su cercanía a un núcleo. El código `-1` marca los puntos que quedan como ruido con esa configuración.

Una cuestión de campo es qué distancia tiene sentido con nuestro espaciamiento. Aquí probamos 17 m como radio didáctico y observamos la sensibilidad al cambiarlo.

In [ ]:
radio = 17
densidad = DBSCAN(eps=radio, min_samples=5).fit(X_ajuste)
etiquetas_db = densidad.labels_
i = densidad.core_sample_indices_[0]
vecinos = np.linalg.norm(X_ajuste - X_ajuste[i], axis=1) <= radio
print(f"Vecindario de la fila {i}: {vecinos.sum()} puntos, incluido el propio")
print(pd.Series(etiquetas_db).value_counts().sort_index().rename("Filas"))

El `DBSCAN` de scikit-learn etiqueta las filas del ajuste y carece de un método `predict` para nuevas filas. Ajustarlo otra vez con datos adicionales puede cambiar los grupos. Por eso el panel de DBSCAN muestra únicamente `X_ajuste`.

In [ ]:
fig, (ax_k, ax_d) = plt.subplots(1, 2, figsize=(10.5, 4.6), sharex=True, sharey=True)
colores = plt.get_cmap("tab10")
for ax, etiquetas in [(ax_k, km.labels_), (ax_d, etiquetas_db)]:
    for g in np.unique(etiquetas):
        puntos = X_ajuste[etiquetas == g]
        ax.scatter(*puntos.T, s=19, color="0.50" if g == -1 else colores(g % 10),
                   marker="x" if g == -1 else "o", label="Ruido" if g == -1 else f"Grupo {g}")
    ax.set(xlabel="Este (m)", aspect="equal")
    ax.grid(alpha=.20)
ax_k.scatter(*centros.T, marker="X", s=95, color="#152C44", label="Centros")
ax_k.scatter(*muestra.T, marker="D", s=75, facecolors="none", edgecolors="black", label="Fila reservada")
ax_k.set(title="K-means · dos centros", ylabel="Norte (m)")
ax_d.add_patch(Circle(X_ajuste[i], radio, fill=False, ls="--", color="#152C44", lw=1.4))
ax_d.set_title(f"DBSCAN · radio de {radio} m")
for ax in (ax_k, ax_d):
    ax.legend(loc="upper center", bbox_to_anchor=(.5, -.16), ncol=2, frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

<a id="parte-4"></a>

### <span style="color:lightgreen">**Comparación y revisión**</span>
***

Para K-means veremos dos criterios internos. La inercia suele bajar al aumentar `k`. La **silueta** compara la distancia de cada fila a su grupo con la distancia al grupo vecino más próximo. Un valor mayor favorece grupos compactos y separados en el espacio de características elegido. Por su geometría, puede dar una lectura distinta de la que obtenemos al mirar trazas curvas.

In [ ]:
resumen_k = []
for k in [2, 3, 4]:
    modelo = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_ajuste)
    resumen_k.append({"k": k, "inercia (m²)": modelo.inertia_,
                      "silueta en ajuste": silhouette_score(X_ajuste, modelo.labels_)})
pd.DataFrame(resumen_k).round(3)

Para DBSCAN, compara cuántos grupos aparecen y cuántos puntos quedan fuera al cambiar `eps`. Los dos modelos resuelven problemas distintos. Una métrica calculada solo para puntos agrupados dejaría fuera las filas etiquetadas como ruido.

In [ ]:
resumen_db = []
for eps in [10, 17, 40]:
    grupos = DBSCAN(eps=eps, min_samples=5).fit_predict(X_ajuste)
    resumen_db.append({"radio (m)": eps,
                       "grupos": np.unique(grupos[grupos != -1]).size,
                       "puntos sin grupo": int((grupos == -1).sum())})
pd.DataFrame(resumen_db)

**Comprueba tu lectura.** ¿Por qué K-means puede separar una misma traza en dos partes? Elige uno de los puntos grises en DBSCAN y explica por qué su etiqueta puede cambiar al aumentar el radio. ¿Qué información de campo pedirías antes de interpretar un grupo como una unidad geológica?

Si quieres comprobar estabilidad en otra campaña, reserva una zona o campaña completa. Para aplicar K-means utiliza los centros aprendidos y el mismo tratamiento de unidades. Si vuelves a ajustar DBSCAN, compara los grupos y las zonas de baja densidad sin exigir que los números de grupo coincidan.

[Guía de clustering de scikit-learn](https://scikit-learn.org/stable/modules/clustering.html)